# Assignment 2: Supernova Hubble Diagram Fitting

**PHYS690: Computational Methods for Physics Research**  
**Units 2-3: Fitting, uncertainty, residuals, and model comparison**

**Student name(s):** _replace this text with your name(s)_  
**Private GitHub repository URL:** _replace this text with your assignment repository link_

## Purpose

This assignment uses a public Type Ia supernova data set to practice the fitting workflow from Lectures 4-8. You will inspect a data table, build a Hubble diagram, fit two reasonable models, diagnose residuals, interpret covariance, and compare models.

The model comparison is deliberately modest. Your goal is not to make a precision cosmology measurement. Your goal is to explain what the data and assumptions support, and what they do not support.

## Scientific setting

Type Ia supernovae are useful standardizable candles: after light-curve corrections, their apparent brightness can be used to estimate distance. A Hubble diagram plots supernova brightness against redshift. At very low redshift, the luminosity distance is approximately proportional to redshift, $d_L \propto z$. At larger redshift, expansion history produces curvature in the distance-redshift relation.

We will use the public Pantheon sample released by Scolnic et al. The table includes corrected apparent magnitudes `mb`, magnitude uncertainties `dmb`, and redshifts `zcmb`. We will ignore the full systematic covariance matrix in this assignment and use the per-object statistical uncertainty column as the fitting uncertainty. That simplification must appear in your final interpretation.

## Deliverable

Submit your private GitHub repository containing a reproducible notebook analysis. Your repository should include:

- this completed notebook;
- a completed command-line script at `scripts/make_supernova_model_comparison.py`;
- final figures saved in `figures/`;
- a concise fit-summary table saved by the script, such as `figures/supernova_fit_summary.csv`;
- a `requirements.txt` file listing the Python packages needed to rerun the notebook and script;
- an updated `README.md` explaining how to rerun the work and where the public data come from;
- a Git history showing meaningful progress through the assignment.

A reasonable repository layout is:

```text
assignment2-supernova-hubble-diagram/
  .gitignore
  README.md
  requirements.txt
  figures/
    supernova_hubble_fit.png
    supernova_residuals.png
  notebooks/
    assignment2_supernova_hubble_diagram.ipynb
  scripts/
    make_supernova_model_comparison.py
```

## Part 1: Set up the Python environment

Open this repository folder in VS Code. Use `Terminal > New Terminal` to create and activate a local Python environment, then install the packages used here.

On macOS:

```bash
python3 -m venv .venv
source .venv/bin/activate
python -m pip install -r requirements.txt
```

On Windows Git Bash:

```bash
python -m venv .venv
source .venv/Scripts/activate
python -m pip install -r requirements.txt
```

After that, open this notebook in VS Code and select the `.venv` Python kernel. Then run the next cell.

In [ ]:
%matplotlib inline

from pathlib import Path
import sys

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
from scipy.optimize import curve_fit
from scipy.stats import chi2

plt.rcParams.update({
    "figure.figsize": (7, 4),
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 11,
})

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
FIGURE_DIR = PROJECT_ROOT / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Python: {sys.version.split()[0]}")
print(f"NumPy: {np.__version__}")
print(f"pandas: {pd.__version__}")
print(f"Matplotlib: {matplotlib.__version__}")
print(f"SciPy: {scipy.__version__}")

## Part 2: Load and inspect the public data

The Pantheon data release is available on GitHub. The file `lcparam_full_long.txt` contains redshifts and corrected apparent magnitudes. The columns we need are:

- `name`: supernova identifier;
- `zcmb`: redshift in the cosmic microwave background frame;
- `mb`: corrected apparent magnitude-like quantity;
- `dmb`: uncertainty on `mb`.

The notebook reads directly from the public URL so the starter repository can stay clean.

In [ ]:
PANTHEON_URL = "https://raw.githubusercontent.com/dscolnic/Pantheon/master/lcparam_full_long.txt"
PANTHEON_COLUMNS = [
    "name", "zcmb", "zhel", "dz", "mb", "dmb", "x1", "dx1", "color",
    "dcolor", "3rdvar", "d3rdvar", "cov_m_s", "cov_m_c", "cov_s_c",
    "set", "ra", "dec",
]

# TODO: Read the public Pantheon table into a DataFrame named raw_supernovae.
#
# Complete instructions:
# 1. Use pandas.read_csv.
# 2. Use PANTHEON_URL as the input path.
# 3. The file is whitespace-separated, so pass sep=r"\s+".
# 4. The first line is a header line from the data release. For this file,
#    use skiprows=1 and names=PANTHEON_COLUMNS. The header includes one
#    extra trailing label, so assigning the actual 18 data columns explicitly
#    is the safest option.
# 5. Display the first few rows with .head().
#
# Course reference: Lecture 3 used pandas.read_csv for tabular data.
# Documentation: https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html
raw_supernovae = ...

raw_supernovae.head()


In [ ]:
# TODO: Inspect the loaded table.
#
# Complete instructions:
# 1. Print the DataFrame shape so you know how many rows and columns were read.
# 2. Use .describe() on the columns ["zcmb", "mb", "dmb"].
# 3. Check that redshift, magnitude, and magnitude uncertainty are numeric.
#
# Course reference: Assignment 1 and Lecture 3 both used .describe()
# to summarize a pandas DataFrame before analysis.
...


### Interpretation prompt

In a markdown cell below this one, briefly answer:

- How many supernovae are in the full table?
- What is the approximate redshift range?
- Why is it useful that the table includes an uncertainty column?

_Your answer here._

## Part 3: Choose an analysis sample

We will focus on supernovae with `0.01 <= zcmb <= 0.30`. The lower cut avoids extremely nearby objects where peculiar velocities can dominate over smooth Hubble expansion. The upper cut keeps the cosmographic approximation simple enough for this assignment.

These cuts are analysis choices. They should be reported, and you should not silently change them later to improve the answer.

In [ ]:
Z_MIN = 0.01
Z_MAX = 0.30

# TODO: Build the analysis DataFrame named analysis_data.
#
# Complete instructions:
# 1. Start from raw_supernovae.
# 2. Keep only rows with Z_MIN <= zcmb <= Z_MAX.
#    Hint: pandas Series have a .between(low, high) method.
# 3. Require finite mb and dmb values using np.isfinite.
# 4. Require dmb > 0 so every selected point has a valid uncertainty.
# 5. Copy the selected rows with .copy().
# 6. Sort by zcmb and reset the index.
# 7. Print how many supernovae remain.
# 8. Display columns ["name", "zcmb", "mb", "dmb"] for the first few rows.
#
# Course reference: Lecture 4 introduced the idea that uncertainty values
# enter the weighted chi-square, so invalid or nonpositive uncertainties
# must be removed before fitting.
analysis_data = ...

...


In [ ]:
# TODO: Make a first Hubble diagram visualization.
#
# Complete instructions:
# 1. Create a figure and axis with plt.subplots().
# 2. Plot mb versus zcmb with vertical error bars from dmb.
#    Hint: use ax.errorbar(..., yerr=..., fmt=".").
# 3. Use a logarithmic x-axis with ax.set_xscale("log").
# 4. Label the axes and add a title.
# 5. Add a legend and call plt.show().
#
# Course reference: Lecture 4 plotted measured data with uncertainty bars
# before fitting. Matplotlib errorbar documentation:
# https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.errorbar.html
fig, ax = ...
...
plt.show()


### Interpretation prompt

In a markdown cell below this one, explain why the Hubble diagram is plotted with a logarithmic redshift axis. What pattern do you expect for magnitude as distance increases?

_Your answer here._

## Part 4: Define two models

We will compare two models for the corrected apparent magnitude as a function of redshift. In both models, the intercept absorbs the unknown combination of absolute supernova magnitude and Hubble constant. Therefore, do not interpret the intercept alone as a measurement of $H_0$.

**Model A: low-redshift Hubble law**

$$m_b(z) = a + 5\log_{10}(z).$$

**Model B: cosmographic extension**

$$m_b(z) = a + 5\log_{10}\left[z\left(1 + \frac{1-q_0}{2}z\right)\right].$$

The extra parameter $q_0$ changes the curvature of the Hubble diagram. Model A is nested inside Model B at $q_0 = 1$. The symbol $q_0$ resembles the cosmological deceleration parameter, but this simplified fit ignores important ingredients such as systematic covariance and higher-redshift expansion terms. Treat it as a shape parameter for this assignment.

In [ ]:
# TODO: Define the two model functions and two diagnostic helper functions.
#
# Complete instructions for hubble_law_magnitude:
# - The function should accept redshift and intercept.
# - Return intercept + 5 * log10(redshift).
#
# Complete instructions for cosmographic_magnitude:
# - The function should accept redshift, intercept, and q0.
# - Define distance_shape = redshift * (1 + 0.5 * (1 - q0) * redshift).
# - Return intercept + 5 * log10(distance_shape).
#
# Complete instructions for normalized_residuals:
# - Accept data, model_function, and parameters.
# - Evaluate model_function(data["zcmb"], *parameters).
# - Return (data["mb"] - prediction) / data["dmb"].
#
# Complete instructions for goodness_of_fit:
# - Compute normalized residuals.
# - Compute chi2 as the sum of squared normalized residuals.
# - Compute ndf = number of data points - number of fitted parameters.
# - Return a dictionary with chi2, ndf, reduced_chi2, and p_value.
# - Use scipy.stats.chi2.sf(chi2_value, ndf) for the upper-tail p-value.
#
# Course references:
# - Lecture 4: residuals, normalized residuals, and chi-square.
# - Lecture 7: reduced chi-square and goodness-of-fit p-values.
# SciPy chi-square docs: https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.chi2.html
def hubble_law_magnitude(redshift, intercept):
    ...


def cosmographic_magnitude(redshift, intercept, q0):
    ...


def normalized_residuals(data, model_function, parameters):
    ...


def goodness_of_fit(data, model_function, parameters):
    ...


## Part 5: Fit both models with weighted least squares

Use `curve_fit` with `sigma=analysis_data["dmb"]` and `absolute_sigma=True`. This tells SciPy that the magnitude uncertainties are the absolute measurement uncertainties used in the weighted least-squares objective

$$\chi^2 = \sum_i \left[\frac{m_{b,i} - m_b(z_i;\theta)}{\sigma_{m,i}}\right]^2.$$

For the cosmographic model, use a finite bound on $q_0$ so the quantity inside the logarithm stays positive over the selected redshift range.

In [ ]:
x = analysis_data["zcmb"].to_numpy()
y = analysis_data["mb"].to_numpy()
sigma_y = analysis_data["dmb"].to_numpy()

# TODO: Fit Model A with scipy.optimize.curve_fit.
#
# Complete instructions:
# 1. Call curve_fit with hubble_law_magnitude as the model.
# 2. Pass x and y as the independent and dependent data.
# 3. Use p0=[25.0] as a reasonable first guess for the intercept.
# 4. Pass sigma=sigma_y and absolute_sigma=True.
# 5. Store the returned best-fit parameters and covariance matrix as
#    hubble_popt and hubble_pcov.
#
# Course reference: Lecture 4 Part 4 used curve_fit with p0, sigma,
# and absolute_sigma=True.
# SciPy docs: https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.curve_fit.html
hubble_popt, hubble_pcov = ...

# TODO: Fit Model B with scipy.optimize.curve_fit.
#
# Complete instructions:
# 1. Call curve_fit with cosmographic_magnitude as the model.
# 2. Use p0=[25.0, -0.5] as starting values for intercept and q0.
# 3. Pass sigma=sigma_y and absolute_sigma=True.
# 4. Use bounds=([0.0, -10.0], [50.0, 3.0]) to keep the logarithm well behaved.
# 5. Store the returned values as cosmo_popt and cosmo_pcov.
#
# Course reference: Lecture 5 discussed why bounds can help prevent
# unphysical or nonconvergent fits.
cosmo_popt, cosmo_pcov = ...

# TODO: Build a fit_summary DataFrame.
#
# Complete instructions:
# 1. For the Hubble-law model, report the intercept, its uncertainty from
#    sqrt(hubble_pcov[0, 0]), and the values returned by goodness_of_fit.
# 2. For the cosmographic model, report the intercept, q0, their covariance
#    uncertainties, and the values returned by goodness_of_fit.
# 3. Display the table.
#
# Course reference: Lecture 4 Part 5 used sqrt(diag(pcov)) to get
# one-standard-deviation parameter uncertainties.
fit_summary = ...

fit_summary


### Interpretation prompt

In a markdown cell below this one, report the fitted parameters with uncertainties. Which model has the lower $\chi^2$? Why is that fact alone not enough to choose the more flexible model?

_Your answer here._

## Part 6: Visualize the fitted models

Plot the data and both fitted curves. Save the result in `figures/supernova_hubble_fit.png`.

In [ ]:
# TODO: Plot the data and both fitted model curves.
#
# Complete instructions:
# 1. Create z_grid with np.linspace from the minimum to maximum selected zcmb.
# 2. Plot the data using ax.errorbar with yerr=sigma_y.
# 3. Plot hubble_law_magnitude(z_grid, *hubble_popt).
# 4. Plot cosmographic_magnitude(z_grid, *cosmo_popt).
# 5. Use a logarithmic x-axis, labels, title, and legend.
# 6. Save the figure to FIGURE_DIR / "supernova_hubble_fit.png" with dpi=200.
#
# Course reference: Lecture 4 showed data plus best-fit model curves.
z_grid = ...

fig, ax = ...
...

fit_figure_path = FIGURE_DIR / "supernova_hubble_fit.png"
# TODO: save the figure here.
...
plt.show()

print(f"Saved {fit_figure_path}")


## Part 7: Residual analysis

Residual plots can show structure that a single $\chi^2$ number hides. Compute normalized residuals for both models and plot them against redshift. Save the result in `figures/supernova_residuals.png`.

In [ ]:
# TODO: Compute and plot normalized residuals for both models.
#
# Complete instructions:
# 1. Add a column named "hubble_residual" to analysis_data using
#    normalized_residuals(..., hubble_law_magnitude, hubble_popt).
# 2. Add a column named "cosmo_residual" using the cosmographic model.
# 3. Create two vertically stacked subplots with shared x and y axes.
# 4. In each panel, draw a horizontal line at residual = 0.
# 5. Scatter plot residuals versus zcmb for one model per panel.
# 6. Use a logarithmic x-axis.
# 7. Save the result to FIGURE_DIR / "supernova_residuals.png".
#
# Course references:
# - Lecture 4: residuals and normalized residuals.
# - Lecture 7: residual comparison and why chi-square alone can hide patterns.
analysis_data["hubble_residual"] = ...
analysis_data["cosmo_residual"] = ...

fig, axes = ...
...

residual_figure_path = FIGURE_DIR / "supernova_residuals.png"
# TODO: save the figure here.
...
plt.show()

print(f"Saved {residual_figure_path}")


### Interpretation prompt

In a markdown cell below this one, compare the residual patterns. Does one model remove a systematic trend? Are there outliers? What would you want to check before claiming a cosmological discovery?

_Your answer here._

## Part 8: Covariance and parameter correlation

For the two-parameter cosmographic model, the covariance matrix describes the local uncertainty ellipse near the best fit. The diagonal entries are variances. The off-diagonal entry is the covariance between the intercept and $q_0$.

Convert the covariance matrix into a correlation matrix and interpret the sign and size of the off-diagonal correlation.

In [ ]:
# TODO: Convert the cosmographic covariance matrix into uncertainties
# and a correlation matrix.
#
# Complete instructions:
# 1. Set parameter_names = ["intercept", "q0"].
# 2. Compute cosmo_uncertainties = sqrt(diag(cosmo_pcov)).
# 3. Compute the correlation matrix using
#    correlation_ij = covariance_ij / (sigma_i * sigma_j).
#    Hint: np.outer(cosmo_uncertainties, cosmo_uncertainties).
# 4. Put the covariance matrix and correlation matrix into pandas DataFrames
#    with parameter names as both index and columns.
# 5. Display both tables.
#
# Course references:
# - Lecture 4 Part 5: covariance and parameter correlation.
# - Lecture 5: covariance/correlation as diagnostics for weakly identified parameters.
parameter_names = ...
cosmo_uncertainties = ...
cosmo_correlation = ...

covariance_table = ...
correlation_table = ...

...


### Interpretation prompt

In a markdown cell below this one, answer:

- What are the one-standard-deviation covariance-based uncertainties on the two parameters?
- Are the intercept and $q_0$ positively or negatively correlated?
- In physical terms, why might shifting the intercept compensate for changing the curvature parameter?

_Your answer here._

## Part 9: Goodness of fit and nested model comparison

Goodness of fit asks whether one model is statistically compatible with the data and uncertainties. Model comparison asks whether the improvement from a more flexible model is large enough to justify the extra parameter.

Because Model A is nested inside Model B at $q_0=1$, compute

$$\Delta\chi^2 = \chi^2_\mathrm{Hubble} - \chi^2_\mathrm{cosmographic}.$$

Under regular conditions, $\Delta\chi^2$ can be compared with a chi-square distribution with $\Delta k = 1$ degree of freedom. Here that approximation is useful, but not perfect: the model is nonlinear, the data have ignored systematic covariance, and the analysis range was chosen by hand.

In [ ]:
# TODO: Compute goodness-of-fit and nested-model comparison quantities.
#
# Complete instructions:
# 1. Use goodness_of_fit for the Hubble-law model and cosmographic model.
# 2. Compute delta_chi2 = chi2_hubble - chi2_cosmographic.
# 3. Compute delta_parameters = number of cosmographic parameters minus
#    number of Hubble-law parameters.
# 4. Compute delta_chi2_p_value = chi2.sf(delta_chi2, delta_parameters).
# 5. Build and display a comparison_table with chi2, ndf, goodness-of-fit
#    p-values, Delta chi2, Delta parameters, and the Delta-chi-square p-value.
#
# Course reference: Lecture 7 Part 4 covered nested hypotheses and
# Delta chi-square tests. Remember that the usual chi-square calibration
# depends on regularity assumptions.
hubble_gof = ...
cosmo_gof = ...

delta_chi2 = ...
delta_parameters = ...
delta_chi2_p_value = ...

comparison_table = ...

comparison_table


### Interpretation prompt

In a markdown cell below this one, write a short model-comparison paragraph. Your paragraph must explain:

- what the goodness-of-fit p-values say about each model;
- what the $\Delta\chi^2$ comparison says about adding $q_0$;
- why this comparison does not establish a precision value of the cosmological deceleration parameter.

_Your answer here._

## Part 10: Final scientific summary

Write a final summary paragraph below. It should be understandable to someone who has not read your code. Include:

- what data set you used;
- what redshift range you analyzed;
- which two models you compared;
- which model better describes the residuals and $\chi^2$;
- what the comparison does establish;
- what it does not establish because of approximations, ignored systematics, or model limitations;
- any substantial use of generative AI, if applicable.

_Your final summary here._

## Part 11: Write a command-line reproduction script

A reproducible analysis should not depend only on notebook state. Complete `scripts/make_supernova_model_comparison.py` so it reproduces the final model-comparison outputs from the command line.

Your script should run from the repository root with:

```bash
python scripts/make_supernova_model_comparison.py
```

The script must:

- read the public Pantheon table from the same URL used in the notebook;
- apply the same `0.01 <= zcmb <= 0.30` selection;
- define the same Hubble-law and cosmographic models;
- fit both models with `curve_fit`;
- compute a concise fit-summary table with parameter estimates, uncertainties, chi-square, degrees of freedom, goodness-of-fit p-values, and the nested-model comparison;
- save `figures/supernova_hubble_fit.png`;
- save `figures/supernova_residuals.png`;
- save a fit-summary table such as `figures/supernova_fit_summary.csv`.

This is the same idea as the command-line reproduction script from Assignment 1: the notebook is for exploration and explanation, while the script is for repeatable production of final outputs. Use functions in the script so the workflow is readable. The scaffolded comments in the script point back to the notebook parts and lecture examples you need.

## Part 12: Update README and submit

Update `README.md` so another person can rerun your work. Include the public data URL, setup instructions, the notebook name, the command to run the script, the names of generated figures, and a concise interpretation of your model comparison.

Suggested final commands:

```bash
git status
git add README.md requirements.txt figures/*.png scripts/make_supernova_model_comparison.py notebooks/assignment2_supernova_hubble_diagram.ipynb
git commit -m "Complete supernova Hubble diagram analysis"
git status
git log --oneline --max-count=5
git push -u origin submission
```

Finally, submit a pull request from the `submission` branch into `main` in your private assignment repository.

## References

- D. M. Scolnic et al., *The Complete Light-curve Sample of Spectroscopically Confirmed SNe Ia from Pan-STARRS1 and Cosmological Constraints from the Combined Pantheon Sample*, Astrophysical Journal 859, 101 (2018), arXiv: [1710.00845](https://arxiv.org/abs/1710.00845).
- Full Pantheon data-release GitHub repository: https://github.com/dscolnic/Pantheon
- SciPy `curve_fit` documentation: https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.curve_fit.html
- SciPy chi-square distribution documentation: https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.chi2.html